# U-Net 6 kênh — Lê Xuân Lộc (23001533)
Huấn luyện trên cặp ảnh EBD pre/post và mask 5 lớp. Chọn **Runtime → Change runtime type → GPU** trước khi chạy. Upload mã nguồn và dữ liệu vào cùng một thư mục Drive; giữ nguyên `splits.json` để so sánh với các mô hình khác.


In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/Thi_giac_may_tinh')  # Sửa theo thư mục của bạn
assert (PROJECT_DIR / 'unet6').is_dir(), 'Cần chép thư mục unet6 vào PROJECT_DIR'
os.chdir(PROJECT_DIR)
print('Project:', Path.cwd())


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
import torch
print('PyTorch:', torch.__version__)
assert torch.cuda.is_available(), 'Hãy chọn runtime GPU trước khi huấn luyện'
print('GPU:', torch.cuda.get_device_name(0))


## 1. Kiểm tra dữ liệu và bộ chia dùng chung
Giữ cùng `splits.json` cho U-Net, Siamese U-Net và DeepLabV3+. Không tạo lại bộ chia giữa các thí nghiệm.

In [ ]:
import json
if not Path('splits.json').exists():
    subprocess.run([sys.executable, '-m', 'unet6.prepare', '--root', '.', '--output', 'splits.json'], check=True)
splits = json.loads(Path('splits.json').read_text())
print(json.dumps(splits['statistics'], indent=2))


## 2. Huấn luyện
Mặc định: 40 epoch, ảnh 512×512, U-Net base=32, batch=4, weighted cross entropy. Khi thiếu VRAM, giảm batch. Mỗi lần chạy đặt thư mục kết quả mới. Chọn mô hình bằng validation mIoU các lớp 1–4, giữ test cho đánh giá cuối.

In [ ]:
from datetime import datetime
RUN_DIR = Path('runs') / ('unet6_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
subprocess.run([sys.executable, '-m', 'unet6.train', '--root', '.',
    '--splits', 'splits.json', '--output', str(RUN_DIR),
    '--epochs', '40', '--batch-size', '4', '--base', '32',
    '--size', '512', '--weighted-loss', '--device', 'cuda'], check=True)
print('Kết quả:', RUN_DIR)


In [ ]:
from IPython.display import display, Image as DisplayImage
display(DisplayImage(filename=str(RUN_DIR / 'learning_curves.png')))
print(json.dumps(json.loads((RUN_DIR / 'best_val_metrics.json').read_text()), indent=2))


## 3. Đánh giá test
Chạy sau khi đã chốt cấu hình bằng validation. Các chỉ số là theo pixel; `null` chỉ lớp không xuất hiện cả trong nhãn thật lẫn dự đoán.

In [ ]:
subprocess.run([sys.executable, '-m', 'unet6.evaluate', '--root', '.',
    '--splits', 'splits.json', '--checkpoint', str(RUN_DIR / 'best.pt'),
    '--output', str(RUN_DIR / 'test'), '--device', 'cuda'], check=True)
import pandas as pd
result = json.loads((RUN_DIR / 'test' / 'metrics.json').read_text())
display(pd.DataFrame(result['per_class']))
print({k: result[k] for k in ['miou_all','miou_damage','macro_dice_all','macro_dice_damage']})
for image in sorted((RUN_DIR / 'test').glob('*_comparison.png'))[:4]:
    display(DisplayImage(filename=str(image)))

subprocess.run([sys.executable, '-m', 'unet6.summarize', '--run', str(RUN_DIR)], check=True)


## 4. Minh họa dự đoán
Nhãn 0 nền, 1 không hư hại, 2 nhẹ, 3 nặng, 4 phá hủy. Số pixel theo lớp không tương đương số công trình.

In [ ]:
from unet6.data import read_mask
import numpy as np
# Chọn mẫu có đủ lớp trong ground truth để minh họa, không chọn theo điểm dự đoán.
sample = next((r for r in splits['test'] if len(np.unique(read_mask(r['mask']))) == 5), splits['test'][0])
subprocess.run([sys.executable, '-m', 'unet6.predict',
    '--pre', sample['pre'], '--post', sample['post'],
    '--checkpoint', str(RUN_DIR / 'best.pt'), '--output', str(RUN_DIR / 'demo'),
    '--device', 'cuda'], check=True)
display(DisplayImage(filename=str(RUN_DIR / 'demo' / 'comparison.png')))
display(DisplayImage(filename=str(RUN_DIR / 'demo' / 'overlay.png')))


## Bàn giao
Gửi `best.pt`, `config.json`, `history.csv`, `test/per_class.csv`, `test/metrics.json`, ảnh so sánh và `splits.json`. Đọc `BAO_CAO_LE_XUAN_LOC.md` để hoàn thiện phần nhận xét dựa trên kết quả thật. Do dữ liệu hiện tại chỉ thuộc HURRICANE-IRMA, không kết luận khả năng tổng quát hóa sang sự kiện khác.